In [0]:
spark.spasparkssassssadasddsadsdafrom pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

spark = SparkSession.builder.appName("RankVariance").getOrCreate()

# Dummy data: fb_comments_count
comments_data = [
    (101, "2019-12-05", 15),
    (102, "2019-12-10", 8),
    (103, "2019-12-15", 20),
    (101, "2020-01-10", 12),
    (102, "2020-01-20", 10),
    (104, "2019-12-01", 25),
    (104, "2020-01-05", 30),
    (105, "2019-12-20", 5),
    (105, "2020-01-15", 6),
]

# Dummy data: fb_active_users
users_data = [
    (101, "Alice", "active", "USA"),
    (102, "Bob", "active", "Canada"),
    (103, "Charlie", "active", "USA"),
    (104, "David", "active", "UK"),
    (105, "Eve", "active", "Canada"),
    (106, "Frank", "active", "Germany"),  # no comments
]

comments_df = spark.createDataFrame(comments_data, ["user_id", "created_at", "number_of_comments"])
users_df = spark.createDataFrame(users_data, ["user_id", "name", "status", "country"])

# Convert created_at to date type
comments_df = comments_df.withColumn("created_at", to_date(col("created_at")))

comments_df.createOrReplaceTempView("fb_comments_count")
users_df.createOrReplaceTempView("fb_active_users")

In [0]:
"""
Compare the total number of comments made by users in each country during December 2019 and January 2020.


For each month, rank countries by their total number of comments in descending order. Countries with the same total should share the same rank, and the next rank should increase by one (without skipping numbers).


Return the names of the countries whose rank improved from December to January (that is, their rank number became smaller).
"""

In [0]:
spark.sql("""
with ranking_dec_19 as 
(
    select country ,
    DENSE_RANK() OVER(order by SUM(number_of_comments) DESC) as dec_19_rank
    from fb_comments_count join fb_active_users using(user_id)
    where created_at BETWEEN '2019-12-01' AND '2019-12-31'
    group by country
),

ranking_jan_20 as 
(
    select country ,
    DENSE_RANK() OVER(order by SUM(number_of_comments) DESC) as jan_20_rank
    from fb_comments_count join fb_active_users using(user_id)
    where created_at BETWEEN '2020-01-01' AND '2020-01-31'
    group by country
)

select country from ranking_dec_19 JOIN ranking_jan_20 using(country)
where jan_20_rank < dec_19_rank         
""").display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, month, year, when
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

# Filter for December 2019
dec_2019 = comments_df.filter(
    (col("created_at") >= "2019-12-01") & (col("created_at") <= "2019-12-31")
).join(
    users_df, on="user_id", how="inner"
).groupBy("country").agg(
    spark_sum("number_of_comments").alias("total_comments_dec")
)

# Add dense rank for December
window_spec_dec = Window.orderBy(col("total_comments_dec").desc())
dec_ranked = dec_2019.withColumn("dec_19_rank", dense_rank().over(window_spec_dec))

# Filter for January 2020
jan_2020 = comments_df.filter(
    (col("created_at") >= "2020-01-01") & (col("created_at") <= "2020-01-31")
).join(
    users_df, on="user_id", how="inner"
).groupBy("country").agg(
    spark_sum("number_of_comments").alias("total_comments_jan")
)

# Add dense rank for January
window_spec_jan = Window.orderBy(col("total_comments_jan").desc())
jan_ranked = jan_2020.withColumn("jan_20_rank", dense_rank().over(window_spec_jan))

# Join and filter for improved rank (lower number = better)
result = dec_ranked.join(
    jan_ranked, on="country", how="inner"
).filter(
    col("jan_20_rank") < col("dec_19_rank")
).select(
    "country",
    "dec_19_rank",
    "jan_20_rank",
    "total_comments_dec",
    "total_comments_jan"
).orderBy("country")

result.display()